# Step 5–7: Inference, Evaluation and Gradio Demo

This notebook runs the full inference pipeline using our fine-tuned T5 model.

**Pipeline:**
1. Extract all comments from code (Python `#`, `"""` or C++ `//`, `/* */`)
2. Fix each comment using T5
3. Rebuild the code with corrected comments
4. Display changes made

In [ ]:
#Load our fine-tuned T5 model from local folder. T5 was trained on kaggle GPU - see kaggle_t5_training.ipynb
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer

MODEL_PATH = r"C:\Users\thiru\autocorrect_comments\t5_trained"

print("Loading T5 tokenizer...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_PATH)

print("Loading T5 model...")
model  = T5ForConditionalGeneration.from_pretrained(MODEL_PATH)
device = torch.device("cpu")
model  = model.to(device)
model.eval()

print("T5 model loaded successfully!")
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def correct_with_t5(text):
    """
    Given any text with typos, T5 corrects it directly.
    No masking needed — T5 sees the full corrupted text.
    """
    # Add the "fix: " prefix — this is how we trained it
    input_text = "fix: " + text
    
    # Tokenize
    input_enc = tokenizer(
        input_text,
        return_tensors = "pt",
        max_length     = 128,
        truncation     = True
    ).to(device)
    
    # Generate correction
    with torch.no_grad():
        output_ids = model.generate(
            input_enc["input_ids"],
            max_length    = 128,
            num_beams     = 4,
            early_stopping= True,
        )
    
    # Decode output
    corrected = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return corrected


# Test on various examples
test_comments = [
    "Retuns the usr from databse by ID",
    "Computes the waighted averege of all valus",
    "Get all chid foleder of a folder",
    "Initalize the databse conection with given credentals",
    "Parses the configuartion file and validtes required filds",
]

print("T5 CORRECTION TEST")
print("=" * 60)
for comment in test_comments:
    corrected = correct_with_t5(comment)
    print(f"Original : {comment}")
    print(f"Corrected: {corrected}")
    print("-" * 60)

In [ ]:
import re
from spellchecker import SpellChecker

spell = SpellChecker()

def find_typo(comment):
    """Find first suspicious word in comment — used for highlighting changes."""
    tokens = comment.split()
    for idx, word in enumerate(tokens):
        clean = word.strip(".,!?:;\"'()")
        if len(clean) < 4:
            continue
        if "_" in clean or clean.isupper():
            continue
        if idx > 0 and clean[0].isupper():
            continue
        misspelled = spell.unknown([clean.lower()])
        if misspelled:
            return clean, idx
    return None, None

In [ ]:
def extract_comments(code, language="python"):
    """Extract all comment lines from code with their positions."""
    lines   = code.splitlines()
    results = []
    
    if language == "python":
        in_docstring   = False
        docstring_char = None
        
        for i, line in enumerate(lines):
            stripped = line.strip()
            found_docstring = False
            
            for quote in ['"""', "'''"]:
                if stripped.startswith(quote):
                    found_docstring = True
                    if not in_docstring:
                        in_docstring   = True
                        docstring_char = quote
                        text = stripped[3:].strip()
                        if stripped.endswith(quote) and len(stripped) > 6:
                            text         = stripped[3:-3].strip()
                            in_docstring = False
                        if text:
                            results.append({
                                "line_idx"    : i,
                                "original"    : line,
                                "comment_text": text,
                                "prefix"      : line[:len(line)-len(line.lstrip())] + quote,
                            })
                    else:
                        in_docstring = False
                    break
            
            if not found_docstring and in_docstring:
                text = stripped
                if text and not text.startswith(":"):
                    results.append({
                        "line_idx"    : i,
                        "original"    : line,
                        "comment_text": text,
                        "prefix"      : line[:len(line)-len(line.lstrip())],
                    })
            
            elif not found_docstring and not in_docstring and stripped.startswith("#"):
                hash_pos = line.index("#")
                text     = line[hash_pos + 1:].strip()
                if text:
                    results.append({
                        "line_idx"    : i,
                        "original"    : line,
                        "comment_text": text,
                        "prefix"      : line[:hash_pos] + "# ",
                    })
    
    elif language == "cpp":
        in_block = False
        
        for i, line in enumerate(lines):
            stripped = line.strip()
            
            if "/*" in line and not in_block:
                in_block   = True
                after_open = line[line.index("/*") + 2:]
                if "*/" in after_open:
                    text     = after_open[:after_open.index("*/")].strip()
                    in_block = False
                else:
                    text = after_open.strip()
                if text:
                    results.append({
                        "line_idx"    : i,
                        "original"    : line,
                        "comment_text": text,
                        "prefix"      : line[:line.index("/*")] + "/* ",
                    })
            
            elif in_block:
                if "*/" in stripped:
                    text     = stripped[:stripped.index("*/")].lstrip("*").strip()
                    in_block = False
                else:
                    text = stripped.lstrip("*").strip()
                if text:
                    results.append({
                        "line_idx"    : i,
                        "original"    : line,
                        "comment_text": text,
                        "prefix"      : line[:len(line)-len(line.lstrip())] + "* ",
                    })
            
            elif stripped.startswith("//"):
                slash_pos = line.index("//")
                text      = line[slash_pos + 2:].strip()
                if text:
                    results.append({
                        "line_idx"    : i,
                        "original"    : line,
                        "comment_text": text,
                        "prefix"      : line[:slash_pos] + "// ",
                    })
    
    return results

In [ ]:
from difflib import SequenceMatcher

# Common programming/technical words to prioritize
TECH_WORDS = {
    'list', 'lists', 'string', 'strings', 'array', 'arrays',
    'result', 'results', 'return', 'returns', 'value', 'values',
    'append', 'insert', 'delete', 'update', 'initialize', 'calculate',
    'database', 'connection', 'average', 'maximum', 'minimum',
    'index', 'length', 'count', 'total', 'output', 'input',
    'file', 'path', 'data', 'type', 'node', 'object', 'class', 'empty',
    'error', 'exception', 'function', 'method', 'parameter',
}

def best_correction(word):
    """
    Pick best correction by combining similarity + tech word priority.
    """
    candidates = spell.candidates(word.lower())
    if not candidates:
        return None
    
    scored = []
    for candidate in candidates:
        sim = SequenceMatcher(None, word.lower(), candidate).ratio()
        # Boost score if candidate is a common tech word
        bonus = 0.15 if candidate in TECH_WORDS else 0
        scored.append((sim + bonus, candidate))
    
    scored.sort(reverse=True)
    best = scored[0][1]
    
    if best != word.lower():
        return best
    return None


# Test again
test_words = [
    ("apend",    "append"),
    ("lits",     "list"),
    ("reslt",    "result"),
    ("initalize","initialize"),
    ("databse",  "database"),
    ("averege",  "average"),
    ("calcualte","calculate"),
    ("studet",   "student"),
    ("fianl",    "final"),
    ("conection","connection"),
]

print("WITH TECH WORD BOOST")
print("=" * 55)
for word, expected in test_words:
    correction = best_correction(word)
    status = "✅" if correction == expected else "❌"
    print(f"{status} {word:15s} → {str(correction):15s} (expected: {expected})")

In [ ]:
from spellchecker import SpellChecker
spell = SpellChecker()

def correct_with_t5_enhanced(comment):
    """
    Stage 1: similarity + tech word boost spell checker
    Stage 2: T5 refines using code context
    """
    tokens = comment.split()
    stage1_tokens = []
    
    for word in tokens:
        clean = word.strip(".,!?:;\"'()")
        punct = word[len(clean):] if word.endswith(tuple(".,!?:;\"'()")) else ""
        
        if len(clean) < 5 or "_" in clean or clean.isupper():
            stage1_tokens.append(word)
            continue
        
        misspelled = spell.unknown([clean.lower()])
        if misspelled:
            suggestion = best_correction(clean)
            if suggestion:
                if clean[0].isupper():
                    suggestion = suggestion.capitalize()
                stage1_tokens.append(suggestion + punct)
            else:
                stage1_tokens.append(word)
        else:
            stage1_tokens.append(word)
    
    stage1_comment = " ".join(stage1_tokens)
    
    # Stage 2 — T5
    input_text = "fix: " + stage1_comment
    input_enc  = tokenizer(
        input_text,
        return_tensors = "pt",
        max_length     = 128,
        truncation     = True
    ).to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_enc["input_ids"],
            max_length     = 128,
            num_beams      = 4,
            early_stopping = True,
        )
    
    corrected = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return corrected, stage1_comment


# Full test
test_comments = [
    "Initalize the databse conection",
    "Retruns the averege of all valus",
    "Calcualte the fianl reslt of the calculaion",
    "Apend the studet marks to the lits",
    "Retuns the usr from databse by ID",
    "Checks if the directroy exists and creats it if not",
]

print("FINAL ENHANCED PIPELINE TEST")
print("=" * 60)
for comment in test_comments:
    final, stage1 = correct_with_t5_enhanced(comment)
    print(f"Input  : {comment}")
    print(f"Stage 1: {stage1}")
    print(f"Final  : {final}")
    print("-" * 60)

In [ ]:
def correct_code_comments(code, language="python"):
    comments = extract_comments(code, language)
    
    if not comments:
        return code, []
    
    lines   = code.splitlines()
    changes = []
    
    for comment in comments:
        original_text          = comment["comment_text"]
        # Use enhanced version now
        corrected_text, stage1 = correct_with_t5_enhanced(original_text)
        
        if corrected_text.strip() != original_text.strip():
            changes.append({
                "line"    : comment["line_idx"] + 1,
                "original": original_text,
                "fixed"   : corrected_text,
            })
            original_line  = comment["original"]
            if comment["prefix"].strip() in ['"""', "'''"]:
                quote          = comment["prefix"].strip()
                corrected_line = comment["prefix"] + corrected_text + quote
            else:
                corrected_line = comment["prefix"] + corrected_text
            lines[comment["line_idx"]] = corrected_line
    
    fixed_code = "\n".join(lines)
    return fixed_code, changes

## Step 7: Gradio Web Demo

Paste any Python or C++ code into the text box.
The model automatically:
1. Finds all comment lines
2. Corrects typos using T5
3. Returns the fixed code with a list of changes

In [ ]:
import gradio as gr

def run_correction(code, language):
    if not code.strip():
        return "", "Please paste some code."
    
    fixed_code, changes = correct_code_comments(code, language.lower())
    
    if not changes:
        changes_text = "✅ No typos found — comments look clean!"
    else:
        changes_text = f"Fixed {len(changes)} comment(s):\n\n"
        for c in changes:
            changes_text += f"Line {c['line']}:\n"
            changes_text += f"  Before: {c['original']}\n"
            changes_text += f"  After : {c['fixed']}\n\n"
    
    return fixed_code, changes_text


demo = gr.Interface(
    fn = run_correction,
    
    inputs = [
        gr.Textbox(
            label       = "Paste your code here (Python or C++)",
            placeholder = """def get_user(id):
    # Retuns the usr from databse
    return db.query(id)""",
            lines = 15,
        ),
        gr.Radio(
            choices = ["Python", "Cpp"],
            value   = "Python",
            label   = "Language",
        ),
    ],
    
    outputs = [
        gr.Textbox(label = "Fixed Code",   lines = 15),
        gr.Textbox(label = "Changes Made", lines = 8),
    ],
    
    title       = "Code Comment Autocorrection",
    description = "Paste your Python or C++ code. The model automatically finds and fixes typos in comments using a two-stage pipeline: spell checker + T5 transformer.",
)

demo.launch(share=False)